In [42]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt

# 1. Parameters / paths

In [43]:
REGION = "wroclaw"

DATA_PATH = f"../data/{REGION}/clean.csv"

PATH_BASINS = f"../data/MPWiK_csv/geometric/Basins.csv"          
PATH_DITCHES = f"../data/MPWiK_csv/geometric/drainage_ditches.csv"   # Zmienione na CSV!
PATH_MANHOLES = f"../data/MPWiK_csv/geometric/stormwater_and_combined_manholes.csv"      
OUTPUT_SPATIAL = f"../data/{REGION}/spacial.csv"

# 2. Setting sectors

In [44]:
df = pd.read_csv(DATA_PATH)
sectors_df = df[['Sektor_ID', 'Lat', 'Lon']].drop_duplicates().copy()

geometry = [Point(lon, lat) for lon, lat in zip(sectors_df['Lon'], sectors_df['Lat'])]
gdf_sectors = gpd.GeoDataFrame(sectors_df, geometry=geometry, crs="EPSG:4326")
gdf_sectors = gdf_sectors.to_crs("EPSG:2177")

# 3. Download data

In [45]:
df_basins_raw = pd.read_csv(PATH_BASINS)
df_basins_raw['geometry'] = df_basins_raw['geometry'].apply(wkt.loads)
gdf_basins = gpd.GeoDataFrame(df_basins_raw, geometry='geometry', crs="EPSG:2177")

df_manholes_raw = pd.read_csv(PATH_MANHOLES)
df_manholes_raw['geometry'] = df_manholes_raw['geometry'].apply(wkt.loads)
gdf_manholes = gpd.GeoDataFrame(df_manholes_raw, geometry='geometry', crs="EPSG:2177")

df_ditches_raw = pd.read_csv(PATH_DITCHES)
df_ditches_raw['geometry'] = df_ditches_raw['geometry'].apply(wkt.loads)
gdf_ditches = gpd.GeoDataFrame(df_ditches_raw, geometry='geometry', crs="EPSG:2177")

# 4. Feature engineering

## - dist to ditch

In [46]:
joined_ditches = gpd.sjoin_nearest(gdf_sectors, gdf_ditches, distance_col="dist_to_ditch", how="left")
gdf_sectors['dist_to_ditch'] = joined_ditches.groupby('Sektor_ID')['dist_to_ditch'].first().values

# - manholes


In [47]:
buffered_sectors = gdf_sectors.copy()
buffered_sectors['geometry'] = buffered_sectors.geometry.buffer(300)

joined_manholes = gpd.sjoin(buffered_sectors, gdf_manholes, how="left", predicate="contains")
manhole_counts = joined_manholes.groupby('Sektor_ID')['index_right'].count().reset_index(name='manholes_300m')

gdf_sectors = gdf_sectors.merge(manhole_counts, on='Sektor_ID', how='left')

## - basins

In [48]:
joined_basins = gpd.sjoin(gdf_sectors, gdf_basins, how="left", predicate="intersects")
gdf_sectors['in_basin'] = joined_basins['index_right'].notna().astype(int).groupby(joined_basins['Sektor_ID']).first().values

# 4. Saving csv

In [49]:
final_spatial_features = pd.DataFrame(gdf_sectors.drop(columns=['geometry']))
final_spatial_features.to_csv(OUTPUT_SPATIAL, index=False)
final_spatial_features.head()

,Sektor_ID,Lat,Lon,dist_to_ditch,manholes_300m,in_basin
0,S_1,50.96,16.71,19204.135493,0,0
1,S_10,51.14,16.71,6812.088247,0,0
2,S_100,51.14,16.83,299.041411,0,0
3,S_101,51.16,16.83,193.312610,0,0
4,S_102,51.18,16.83,218.763942,0,0


In [50]:
final_spatial_features.describe()

,Lat,Lon,dist_to_ditch,manholes_300m,in_basin
count,420.000000,420.000000,420.000000,420.000000,420.0
mean,51.100000,16.980000,4512.630087,26.628571,0.0
std,0.086513,0.161748,4187.453069,80.391139,0.0
min,50.960000,16.710000,0.596424,0.000000,0.0
25%,51.020000,16.845000,672.202800,0.000000,0.0
50%,51.100000,16.980000,3786.952073,0.000000,0.0
75%,51.180000,17.115000,7113.732597,0.000000,0.0
max,51.240000,17.250000,19204.135493,497.000000,0.0
